# Recommenders & Bandits

- [Recommender Systems](#Recommender-Systems)
  - [Two-stage funnel: candidate generation vs. ranking](#Two-stage-funnel:-candidate-generation-vs.-ranking)
  - [Cold-start](#Cold-start)
  - [Multi-objective fusion at serving](#Multi-objective-fusion-at-serving)
  - [Collaborative filtering](#Collaborative-filtering)
  - [Matrix factorization](#Matrix-factorization)
  - [Ranking metrics: NDCG, MAP and recall@k](#Ranking-metrics:-NDCG,-MAP-and-recall@k)
- [Reinforcement Learning and Bandits](#Reinforcement-Learning-and-Bandits)
  - [RL vs. supervised learning](#RL-vs.-supervised-learning)
  - [The multi-armed bandit problem and exploration-exploitation](#The-multi-armed-bandit-problem-and-exploration-exploitation)
  - [Epsilon-greedy](#Epsilon-greedy)
  - [On-policy vs. off-policy](#On-policy-vs.-off-policy)
  - [UCB and Thompson sampling](#UCB-and-Thompson-sampling)
- [Deep Learning Recommenders](#Deep-Learning-Recommenders)
  - [Two-tower model vs. DCN / wide-and-deep](#Two-tower-model-vs.-DCN-/-wide-and-deep)
  - [In-batch negatives and hard-negative mining](#In-batch-negatives-and-hard-negative-mining)
- [Blogs](#Blogs)

### Recommender Systems

This section covers ranking and recommendation at scale: the two-stage candidate-generation-then-ranking funnel, cold-start handling and multi-objective fusion at serving. The deep-learning pieces (two-tower retrieval, in-batch and hard negatives) are in "Deep Learning Recommenders" below. The bi-encoder vs. cross-encoder tradeoff in `09-rag-evals.ipynb` has the same shape (cheap and approximate at scale vs. expensive and precise on a shortlist) in a different domain.

#### Two-stage funnel: candidate generation vs. ranking

Scoring every item for every user with a full cross-feature model does not scale at serving time (10M items x every request). The standard fix is a two-stage funnel:

```
10,000,000 items
      |  stage 1: candidate generation (cheap, approximate, e.g. two-tower + ANN search)
  ~500 candidates
      |  stage 2: ranking (expensive, precise, full cross-feature model, only run on the shortlist)
   ~10 shown to the user
```
Stage 1 optimizes for recall (miss nothing good) over a huge catalog, cheaply. Stage 2 optimizes for precision (get the order right) over a small shortlist, expensively. Getting the split wrong in either direction breaks the pipeline: a stage 1 that drops good candidates, or a stage 2 too slow to run even on 500 items.

#### Cold-start

A brand-new item or user has no interaction history, so its ID-based embedding is untrained (close to its random initialization) and useless for scoring. The standard fallback is a content-based proxy: derive an embedding from item metadata (category, text, image), often by averaging the learned embeddings of similar warm items in the same category.

In [ ]:
import numpy as np

# warm items in the same category, with real interaction-learned embeddings
warm_items_in_category = np.array([
    [0.85, 0.55],
    [0.95, 0.65],
    [0.90, 0.60],
])
category_centroid = warm_items_in_category.mean(axis=0)
print("category centroid, used as the cold item's proxy embedding:", category_centroid)

untrained_id_embedding = np.array([0.02, -0.01])  # what the item's own ID-embedding looks like pre-training
print("vs. the item's own untrained ID-embedding:            ", untrained_id_embedding)

The category-centroid proxy lands close to where the item's own embedding will eventually converge once it has real interactions, so the item gets a reasonable score from day one. The untrained ID embedding carries no signal and would rank the item essentially at random.

#### Multi-objective fusion at serving

Optimizing the ranker purely for click probability tends to surface clickbait and undervalue items with lower click rates but higher downstream value (revenue, long-term engagement, retention). Fusion combines several predicted objectives into one serving-time score, usually a weighted sum: `score = w_click * p_click + w_engage * p_engage + w_value * business_value`. The business sets the weights; they are not learned end to end.

Toy example: three candidates ranked by CTR alone vs. by a fused score with `w_click=1.0, w_engage=1.0, w_value=0.5`:

In [ ]:
candidates = {
    "A": dict(p_click=0.30, p_engage=0.10, value=1.0),
    "B": dict(p_click=0.20, p_engage=0.05, value=5.0),
    "C": dict(p_click=0.15, p_engage=0.20, value=0.5),
}
print("CTR-only ranking:", sorted(candidates, key=lambda k: -candidates[k]["p_click"]))

w_click, w_engage, w_value = 1.0, 1.0, 0.5
fused = {k: w_click*v["p_click"] + w_engage*v["p_engage"] + w_value*v["value"] for k, v in candidates.items()}
print("fused scores:", fused)
print("fused ranking:  ", sorted(fused, key=lambda k: -fused[k]))

**The tradeoff.**

Item B has the lowest click probability but the highest business value, and the fused score correctly promotes it to #1. The cost is that the weights are a real business decision with no single correct answer. Too click-heavy and you starve high-value, lower-CTR items. Too value-heavy and you flood users with high-value items they never engage with. In practice the weights are tuned against a holdout business metric (see `03-ab-causal.ipynb`), not set once and left alone.

#### Collaborative filtering

User-based: find similar users, recommend what they liked. Item-based: find items similar to what user already liked (via co-occurrence patterns) — usually more stable since item-similarity changes less over time than user tastes.

#### Matrix factorization

Decompose sparse user-item rating matrix into low-rank user and item latent-factor matrices (e.g. via SVD or ALS) — predicted rating = dot product of latent vectors. Handles sparsity better than raw similarity.

In [1]:
import numpy as np
R = np.array([[5, 4, 0, 1], [4, 0, 0, 1], [1, 1, 0, 5], [1, 0, 4, 4], [0, 1, 5, 4]], float)   # 0 = not rated
mask = R > 0
rng = np.random.default_rng(0)
U, V = rng.normal(0, 0.1, (5, 2)), rng.normal(0, 0.1, (4, 2))
for _ in range(3000):                                     # gradient descent on observed entries only
    E = mask * (R - U @ V.T)
    U += 0.02 * (E @ V - 0.02 * U); V += 0.02 * (E.T @ U - 0.02 * V)
print("predicted ratings (unrated cells are the predictions):")
print(np.round(U @ V.T, 1))

predicted ratings (unrated cells are the predictions):
[[5.  4.  5.9 1. ]
 [4.  3.2 4.8 1. ]
 [1.1 0.9 5.2 5. ]
 [0.9 0.7 4.1 3.9]
 [1.5 1.2 4.9 4.1]]


#### Ranking metrics: NDCG, MAP and recall@k

Recall@k: share of relevant items in top-k. MAP: mean average precision across queries (binary relevance). NDCG@k: DCG=Σrel_i/log2(i+1) normalized by ideal — graded relevance, position-discounted. Two-tower: user and item encoders → dot product; item vectors precomputed in an ANN index; retrieval stage before the ranker.

In [2]:
import numpy as np
def dcg(rels): return sum((2 ** r - 1) / np.log2(i + 2) for i, r in enumerate(rels))
def ndcg(rels): return dcg(rels) / dcg(sorted(rels, reverse=True))
print("NDCG, relevant items scattered :", round(ndcg([3, 2, 3, 0, 1]), 3))
print("NDCG, best items first         :", round(ndcg([3, 3, 2, 1, 0]), 3))

NDCG, relevant items scattered : 0.957
NDCG, best items first         : 1.0


### Reinforcement Learning and Bandits

This section covers RL as a paradigm distinct from supervised learning, the multi-armed bandit as RL's simplest special case, epsilon-greedy exploration, and the on-policy/off-policy distinction. A bandit is essentially a continuously adapting experiment: it moves traffic toward whichever arm looks best so far instead of holding a fixed 50/50 split for the whole run (see `03-ab-causal.ipynb`).

#### RL vs. supervised learning

In supervised learning, every training example has a fixed, correct label decided in advance and independent of the model's predictions. The loss compares prediction to label directly, and one example's label never depends on a decision made on a previous example.

In reinforcement learning there is no fixed label, only a reward received after taking an action. The action you take now also changes which states and rewards you get to observe later, so the data distribution is a consequence of your own past decisions. This makes RL harder, and it is what bandit and recommender-ranking problems need: showing a user item X instead of item Y costs more than one label, because it changes what you will ever learn about item Y for that user.

#### The multi-armed bandit problem and exploration-exploitation

A bandit has K arms (options), each with an unknown, fixed probability of paying a reward. Each round you pick one arm, observe its reward, and try to maximize cumulative reward. The tension is between **exploit** (play the arm that currently looks best on the limited data so far) and **explore** (try a less-played arm that might be better but lacks data).

Toy setup: 3 arms with hidden true reward probabilities `A=0.3, B=0.5, C=0.2`. The algorithm never sees them; they only generate rewards and let us check performance.

#### Epsilon-greedy

The simplest fix: with probability epsilon pick a random arm (explore), otherwise pick the arm with the highest observed average reward (exploit). Pure greedy (`epsilon=0`) can lock onto a suboptimal arm forever if its first few samples look good by chance, because it never revisits the other arms to find out.

In [ ]:
import numpy as np

true_probs = {"A": 0.3, "B": 0.5, "C": 0.2}  # B is the actual best arm
arms = list(true_probs.keys())

def run_epsilon_greedy(epsilon, n_rounds, seed):
    rng = np.random.default_rng(seed)
    counts = {a: 0 for a in arms}
    sums = {a: 0.0 for a in arms}
    total_reward = 0
    for t in range(n_rounds):
        if t < len(arms):
            a = arms[t]  # try each arm once first
        elif rng.random() < epsilon:
            a = arms[rng.integers(len(arms))]
        else:
            avg = {k: (sums[k] / counts[k] if counts[k] > 0 else 0) for k in arms}
            a = max(avg, key=avg.get)
        r = 1 if rng.random() < true_probs[a] else 0
        counts[a] += 1
        sums[a] += r
        total_reward += r
    return total_reward, counts

for eps in [0.0, 0.1]:
    total_reward, counts = run_epsilon_greedy(eps, n_rounds=300, seed=3)
    print(f"epsilon={eps}: total_reward={total_reward}, arm_counts={counts}, reward_rate={round(total_reward/300, 3)}")

**Reading the result.**

With this random seed, pure greedy (`epsilon=0`) gets unlucky on its first pass. Arm A's single initial sample pays off, so greedy locks onto A and plays it 295 of 300 rounds. It converges to A's true rate of about 0.30 and never discovers that B (true rate 0.50) is better. Epsilon-greedy with `epsilon=0.1`, given the same luck, still spends enough random rounds re-sampling every arm to notice that B is better and shifts most of its plays there (243 of 300), nearly doubling the reward rate to about 0.50. That is the concrete cost of pure exploitation: a small, fixed amount of continued exploration lets the algorithm recover from a bad first impression.

#### On-policy vs. off-policy

**On-policy:** evaluate or improve the same policy that generates the data (the epsilon-greedy above learns from the actions epsilon-greedy itself chose). **Off-policy:** learn about a different target policy from data generated by a behavior policy, for example estimating how a proposed new ranking model would have performed using only logged data from the current production model. Off-policy evaluation needs a correction, because the behavior policy over-represents the actions it favored. An action the target policy prefers much more than the behavior policy did should count for more, not less.

Importance sampling is the standard correction: reweight each observed reward by `target_policy_prob(action) / behavior_policy_prob(action)`. Toy example: the behavior policy took an action with probability 0.2, the target policy would have taken it with probability 0.5, and the observed reward was 10.

In [ ]:
behavior_prob = 0.2
target_prob = 0.5
observed_reward = 10

importance_weight = target_prob / behavior_prob
weighted_reward = observed_reward * importance_weight
print("importance weight:", importance_weight)
print("importance-weighted reward estimate:", weighted_reward)

**Why this matters for offline evaluation.**

The naive estimate uses the raw reward of 10, which undercounts this action's value under the target policy, since it would have chosen the action far more often (0.5 vs. 0.2). The importance-weighted estimate of 25 corrects for that mismatch. This is why off-policy evaluation from historical logs is a viable, much cheaper alternative to running a new online experiment (`03-ab-causal.ipynb`) for every new ranking or bandit policy. The cost is high variance when the two policies diverge a lot: very small `behavior_prob` values blow up the importance weight.

#### UCB and Thompson sampling

Explore–exploit under uncertainty. ε-greedy: random arm w.p. ε else best. UCB: argmax(mean+√(2ln t/n)) — optimism. Thompson: sample each arm's posterior, play the max — Bayesian, usually best empirically. Regret O(log T). Contextual bandits add features.

In [3]:
import numpy as np
rng = np.random.default_rng(0)
p = np.array([0.3, 0.5, 0.2]); T = 3000
def simulate(policy):
    pulls, wins, total = np.zeros(3), np.zeros(3), 0
    for t in range(T):
        if policy == "ucb":
            a = t if t < 3 else int(np.argmax(wins / pulls + np.sqrt(2 * np.log(t) / pulls)))
        else:
            a = int(np.argmax(rng.beta(1 + wins, 1 + pulls - wins)))
        r = rng.random() < p[a]; pulls[a] += 1; wins[a] += r; total += r
    return int(total), pulls.astype(int).tolist()
for policy in ("ucb", "thompson"):
    total, pulls = simulate(policy)
    print(f"{policy:9s}: total reward {total} | pulls per arm {pulls} (best arm is index 1, true rate 0.5)")

ucb      : total reward 1447 | pulls per arm [164, 2696, 140] (best arm is index 1, true rate 0.5)
thompson : total reward 1517 | pulls per arm [93, 2887, 20] (best arm is index 1, true rate 0.5)


### Deep Learning Recommenders

#### Two-tower model vs. DCN / wide-and-deep

**Two-tower:** one neural network encodes the user into a vector and another encodes the item into a vector of the same dimension, and the score is their dot product. Item vectors do not depend on the user, so every item embedding can be pre-computed offline and indexed for approximate nearest-neighbor (ANN) search. That is what makes two-tower a cheap stage-1 candidate generator.

**DCN / wide-and-deep:** one network takes user and item features together and lets them cross (explicit feature crosses or deep interaction layers). It is more accurate because it can learn interactions between specific user and item features, but the score cannot be pre-computed since it depends on both sides jointly, so each candidate is scored at request time. That is why it is the stage-2 ranker, run only on the stage-1 shortlist.

Toy two-tower scoring: user vector `u = [1.0, 0.5]` and three candidate item vectors in the same embedding space:

In [ ]:
import numpy as np

u = np.array([1.0, 0.5])
items = {
    "item_relevant":   np.array([1.0, 0.5]),
    "item_somewhat":   np.array([0.2, 0.1]),
    "item_irrelevant": np.array([-1.0, 0.5]),
}
for name, v in items.items():
    print(name, "dot-product score:", np.dot(u, v))

The relevant item scores highest by construction (its vector is nearly identical to the user's), and the irrelevant item scores negative. This dot-product structure is what ANN indexes (built for cosine or dot-product similarity) can search in sub-linear time. No such shortcut exists for a DCN-style score that mixes user and item features.

#### In-batch negatives and hard-negative mining

Training a two-tower model needs negative (non-matching) user-item pairs, not just the positives in the interaction logs. **In-batch negatives** treat every other item in the training batch as a negative for each user. They are cheap (no extra sampling), but they are whatever happened to be in the batch, usually easy to tell apart from the true positive.

Toy example: 4 users and 4 items, where the diagonal is the true (user, item) pair and every off-diagonal cell is an in-batch negative.

In [ ]:
import numpy as np

U = np.array([
    [1.0, 0.2],
    [0.1, 1.0],
    [0.8, 0.8],
    [-0.5, 0.5],
])
I = np.array([
    [1.0, 0.1],   # true match for user 0
    [0.0, 0.9],   # true match for user 1
    [0.9, 0.7],   # true match for user 2
    [-0.6, 0.4],  # true match for user 3
])
scores = U @ I.T  # row = user, col = item; off-diagonal = in-batch negatives
print("score matrix (rows=users, cols=items):")
print(np.round(scores, 3))

def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()

for row in range(4):
    probs = softmax(scores[row])
    loss = -np.log(probs[row])
    print(f"user {row}: softmax over in-batch items = {np.round(probs, 3)}, loss on true positive = {round(loss, 3)}")

**Why hard negatives matter.**

For user 0, item 2 (score 1.04) outscores item 0, the true positive (score 1.02), even though item 2 belongs to a different user. Item 2 is a **hard negative**: close enough in embedding space to be confusable, and exactly the kind of near-miss that in-batch sampling may not surface reliably (it depends on random batch composition). Hard-negative mining deliberately finds negatives like this (for example the top-scoring non-matches from a previous training round) and adds them to every batch. That sharpens the decision boundary where it matters at serving time, near the top of the ranked list, instead of giving an artificially easy loss from batches full of obviously irrelevant negatives.

### Blogs

- **YouTube, retrieval with in-batch negatives.** YouTube's RecSys 2019 paper trains a two-tower retrieval model with in-batch negatives and notes that this loss is biased toward popular items, because popular items show up as negatives more often. It corrects the bias with an item-frequency estimate computed from streaming data, and used the approach to build Neural Deep Retrieval over a corpus of tens of millions of videos. [Sampling-bias-corrected neural modeling for large corpus item recommendations](https://research.google/pubs/sampling-bias-corrected-neural-modeling-for-large-corpus-item-recommendations/)
- **Pinterest, two-tower retrieval and lightweight scoring.** Pinterest Engineering describes two-tower models as the standard architecture for the retrieval and lightweight ranking stages, for example a learned retrieval system for Home Feed and a unified lightweight scoring stage across candidate generators, ahead of heavier rankers. [Establishing a Large Scale Learned Retrieval System at Pinterest](https://medium.com/pinterest-engineering/establishing-a-large-scale-learned-retrieval-system-at-pinterest-eb0eaf7b92c5), [Pinterest Home Feed Unified Lightweight Scoring: A Two-tower Approach](https://medium.com/pinterest-engineering/pinterest-home-feed-unified-lightweight-scoring-a-two-tower-approach-b3143ac70b55)
- **Netflix, bandits for artwork.** Netflix frames artwork personalization as a contextual bandit problem: the system must explore different images for a title while exploiting what each member responds to, which is the explore/exploit setup from the bandit section above. [Artwork Personalization at Netflix](https://netflixtechblog.com/artwork-personalization-c589f074ad76)

Common pattern: a cheap two-tower retrieval stage over a huge catalog, then heavier ranking, with bandit-style exploration where outcomes are uncertain.